# 00 · Setup & Data

Build a benchmark of **real, public insurance documents** with **hand labels** for an
`ai_parse_document` → `ai_extract` → **MLflow evaluate** pipeline.

1. Create schema + volume.
2. Download **10 real public ACORD / CMS sample PDFs** (genuine, messy layouts — OCR,
   multi-column tables, template placeholders).
3. Write a hand-labelled `golden_labels` table (the labels carry the non-obvious rules:
   **placeholders are not values** — `"Insert Insurer name"`, `"Carrier A"` → null — and
   **primary certificate only** for multi-cert packages).

> Real public certificates of liability carry **policy + coverage** fields, not property
> *underwriting* fields (flood zone, construction, year built, …) — those live on
> declarations pages / SOVs. The same parse→extract→evaluate loop applies to a customer's
> own dec/SOV docs once labelled.

In [0]:
from config import CATALOG, SCHEMA, VOLUME, VOLUME_PATH

# PDFs are staged by fins_data/generate_data.py into the common schema -- verify, don't download.
pdfs = [f.name for f in dbutils.fs.ls(VOLUME_PATH) if f.name.lower().endswith('.pdf')]
assert pdfs, (
    f"No PDFs found in {VOLUME_PATH}. Run fins_data/generate_data.py first -- "
    f"it stages the public insurance PDFs into the common schema."
)
print(f"Found {len(pdfs)} insurance PDFs in {VOLUME_PATH}")

## Hand-labelled golden set

Ground truth for each document. `None` = the field is absent or a placeholder (the extractor should return null). `coverage_types` is a list, scored by recall.

In [0]:
GENERIC_ACORD = ["General Liability", "Automobile Liability", "Umbrella Liability", "Workers Compensation"]
CMS1500_PAYORS = ["Medicare", "Medicaid", "TRICARE", "CHAMPVA"]
ACORD = "ACORD Certificate of Liability Insurance"

golden = [
  dict(filename="acord_certificate.pdf",      document_type=ACORD, insurer_name="Sample company", insured_name="Sample Insured Inc", policy_number="Sample no", effective_date="01/01/2010", expiration_date="01/01/2011", coverage_types=GENERIC_ACORD),
  dict(filename="acord_nyc_dcla.pdf",         document_type=ACORD, insurer_name="US Underwriters Insurance Co", insured_name=None, policy_number="ABCD1234567", effective_date="07/01/2025", expiration_date="06/30/2026", coverage_types=["Commercial General Liability","Automobile Liability","Workers Compensation"]),
  dict(filename="acord_nyc_mome.pdf",         document_type=ACORD, insurer_name=None, insured_name=None, policy_number="123456789", effective_date=None, expiration_date=None, coverage_types=["Commercial General Liability","Automobile Liability","Workers Compensation"]),
  dict(filename="nyc_workerscomp_sample.pdf", document_type="NYS Workers Compensation Certificate", insurer_name="ABC Insurance Company", insured_name=None, policy_number="E 1234567890", effective_date="07/01/2016", expiration_date="06/30/2017", coverage_types=["Workers Compensation"]),
  dict(filename="acord_nyc_dycd.pdf",         document_type=ACORD, insurer_name=None, insured_name=None, policy_number=None, effective_date=None, expiration_date=None, coverage_types=["Commercial General Liability","Automobile Liability","Workers Compensation"]),
  dict(filename="acord_moval.pdf",            document_type=ACORD, insurer_name=None, insured_name=None, policy_number="XX0111222333", effective_date=None, expiration_date=None, coverage_types=["Commercial General Liability","Automobile Liability","Workers Compensation"]),
  dict(filename="acord_ny_ogs.pdf",           document_type=ACORD, insurer_name=None, insured_name=None, policy_number=None, effective_date="04/01/2011", expiration_date="04/01/2012", coverage_types=["Commercial General Liability","Automobile Liability","Umbrella Liability","Workers Compensation"]),
  dict(filename="nyc_acord_sample.pdf",       document_type=ACORD, insurer_name=None, insured_name=None, policy_number=None, effective_date=None, expiration_date=None, coverage_types=GENERIC_ACORD),
  dict(filename="cms1500_claim_form.pdf",     document_type="CMS-1500 Health Insurance Claim Form", insurer_name=None, insured_name=None, policy_number=None, effective_date=None, expiration_date=None, coverage_types=CMS1500_PAYORS),
  dict(filename="acord_tx.pdf",               document_type=ACORD, insurer_name=None, insured_name=None, policy_number=None, effective_date=None, expiration_date=None, coverage_types=GENERIC_ACORD),
]

import pandas as pd
(spark.createDataFrame(pd.DataFrame(golden))
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.golden_labels"))
print("golden_labels rows:", spark.table(f"{CATALOG}.{SCHEMA}.golden_labels").count())
display(spark.table(f"{CATALOG}.{SCHEMA}.golden_labels"))

_Next: `01_parse_and_cost`._